# Motivacion

En el futuro puede que utilicen tanto R como Python para realizar regresiones lineales, y aunque no se realiza exactamente igual, hay una forma de aprender desde Python y que sea haya sintaxis/inputs/outputs similares a R, esto con el objetivo de que cuando requieran cambiar de lenguaje, sea amigable la transicion.

Por ello, haremos uso de la paqueteria _statsmodels_ y no _scikit-learn_, esta ultima es mas robusta, aunque para regresion requiere realizar pasos que en _statsmodels_ nos podemos ahorrar.



# Formula del modelo



Antes de comenzar a usar las funciones para ajustar modelos de regresion, es importante tener algunos aspectos en cuenta, que nos serviran para ambos casos (simple y multiple)


En particular, veremos la forma de indicarle a Python el modelo matematico que deseamos ajustar.




## Datos de ejemplo


Supongamos que tenemos la siguiente base de datos con 4 variables continuas $X_{1},X_{2},X_{3},X_{4}$, una variable nominal $catNum$ de dos niveles ($0, 1$) y una variable nominal $catLetras$ de tres niveles ($A,B,C$)




In [ ]:
library(tidyverse)
set.seed(42)

In [ ]:

n <- 100

# regresoras
variable1 <- runif(n, 0, 100)
variable2 <- runif(n, 0, 50)
variable3 <- runif(n, 0, 30)
variable4 <- runif(n, 0, 10)

# respuesta
respuesta <- 3 + 0.5 * variable1 + 2 * variable2 - 1.5 * variable3 +
  rnorm(n, 0, 5) - 0.2 * variable4


#categoricas
categoriasLetras <- c("A", "B", "C")
catLetras <- sample(categoriasLetras, n, replace = TRUE)

categoriasNum <- c(0, 1)
catNum <- sample(categoriasNum, n, replace = TRUE)

#datos
df <- data.frame(
  X1 = variable1,
  X2 = variable2,
  X3 = variable3,
  X4 = variable4,
  Y = respuesta,
  catLetras = factor(catLetras, levels = categoriasLetras),
  catNum = factor(catNum, levels = categoriasNum)
)

head(df)


,X1,X2,X3,X4,Y,catLetras,catNum
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>
1,64.17455,47.1227846,4.736403,8.784290,126.32801,C,0
2,51.90959,48.1304007,13.269739,9.306049,113.74748,A,1
3,73.65883,36.9927640,29.032010,3.921785,62.59826,C,1
4,13.46666,36.6622953,14.537638,1.588468,55.17949,C,0
5,65.69923,26.7880645,7.573753,3.199476,73.89611,C,1
6,70.50648,0.1136483,7.790699,3.069656,20.91028,A,1


## Modelo matematico y sintaxis en Python

Supongamos que queremos ajustar el modelo de regresion:

$ y = \beta_{0} + \beta_{1}x_{1} $

La "formula" del modelo de regresion que se ajusta, se escribe como:

**Y ~ X1**

Es decir, la formula del modelo se escribe **usando los nombres de las columnas del df**. La variable a la izquierda de "~" es la variable dependiente $y$ y la variable de la derecha es la independiente $x_{1}$. Si quisieramos agregar mas variables basta con agregar "+" antes de cada variable (Y ~ X1 + X2).

Ademas, no es necesario especificar el intercepto $\beta_{0}$, ya que por default se agrega, en caso de querer quitarlo se agrega "-1" a la expresion.
**Quitar el intercepto en la formula es un error**, sin embargo es solo para que sepan como se hace.

Por ejemplo

$ y =\beta_{2}x_{2} $

**Y ~ X2 - 1**


# Ajuste de un modelo de regresion

Una vez que sabemos como se escribe el modelo que deseamos ajustar, el siguiente paso es aprender a crearlo.



## Todas numericas

Supongamos que es de interes ajustar $ y = \beta_{0} + \beta_{1}x_{1} $

La "formula" del modelo de regresion que se ajusta:

**Y ~ X1**

Y se implementa:


In [ ]:
modelo <- lm(Y ~ X1 , data = df)


summary(modelo)



Call:
lm(formula = Y ~ X1, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-64.672 -20.991   1.937  25.281  64.686 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  38.1493     6.7582   5.645 1.61e-07 ***
X1            0.4595     0.1148   4.004 0.000121 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 33.94 on 98 degrees of freedom
Multiple R-squared:  0.1406,	Adjusted R-squared:  0.1318 
F-statistic: 16.04 on 1 and 98 DF,  p-value: 0.000121


Antes de explicar algunas caracteristicas del modelo que acabamos de ajustar como los coeficientes estimados, pruebas de hipotesis, intervalos de confianza, sigamos probando algunas variaciones de las formulas.

## Al menos 1 variable categorica

**Observacion:**
Cuando tenemos una variable categorica de k niveles, en el preprocesamiento se debe indicar que es categorica asi como el orden de los niveles (categorias) aun cuando sea nominal y los niveles no tienen un orden como tal. Realizaremos esto con `factor()` luego de haber creado el dataframe con las variables.
Al indicar el orden de los niveles, a la categoria que este en primer lugar le llamaremos **nivel de referencia** de la variable. Naturalmente si excluimos el nivel de referencia nos quedan k-1 categorias.

El correcto tratamiento de las variables categoricas en los modelos de regresion es crucial, pues es un aspecto que se suele omitir dentro de la teoria pero es comun encontrarnos con datos que contengan variables de esta naturaleza.

Al tener una variable categorica con k categorias, en automatico se realiza un preprocesamiento al excluir al nivel de referencia, y descomponiendo esta variable en k-1 variables binarias que serviran para indicar si esa observacion corresponde a cierta categoria (1) o no corresponde a esa categoria (0), esto sin importar si es nominal u ordinal.

Por ejemplo, supongamos que tenemos observaciones de la variable _Raza_, que tiene **3** categorias: Blanca, Negra y Latina. Y definimos los niveles de nuestra variable con el siguiente orden: (1)Latina, (2)Blanca, (3)Negra.

El preprocesamiento que realiza `lm()` es crear **2** variables indicadoras (0,1) para las categorias que no sean el **nivel de referencia** (Blanca y Negra), y a cada una de estas nuevas variables estimar un coeficiente $\beta_{r}$

Como se puede observar, **es muy importante tener claro cual es el nivel de referencia** de la variable categorica para evitar confusiones con los coeficientes, si cometemos la mala practica de que en el preprocesamiento no indicamos que nuestra variable es categorica y por ende no especificamos el nivel de referencia, R lo elige por nosotros.

A continuacion se muestra este procedimiento de formal manual, para ilustrar mejor el ejemplo. Notemos que pasamos de la variable _Raza_ a las variables binarias _Raza Blanca_ y _Raza Negra_, podemos interpretar que si en ambas variables hay un 0, significa que la observacion corresponde al nivel de referencia de la categoria (Latina)


In [5]:
razas <- c("Blanca", "Negra", "Latina")
df$Raza <- factor(sample(razas, n, replace = TRUE),
                  levels = c("Latina", "Blanca", "Negra"))

raza_dummies <- model.matrix(~ Raza, data = df)
df_with_dummies <- cbind(df["Raza"], raza_dummies[, -1])

head(df_with_dummies)


,Raza,RazaBlanca,RazaNegra
,<fct>,<dbl>,<dbl>
1,Blanca,1,0
2,Negra,0,1
3,Latina,0,0
4,Blanca,1,0
5,Blanca,1,0
6,Latina,0,0


### La variable categorica esta en formato de string

Supongamos que nos interesa ajustar

$y = \beta_{0} + \beta_{2}$**catLetras**  


Notemos que **catLetras** son strings ("A","B","C")

Entonces el modelo se ajusta igual que en el caso de todas numericas, debido a que la funcion identifica que la variable es categorica al estar en formato string (aun cuando no se haya indicado en el preprocesamiento)

In [6]:
modeloCatLetras <- lm(Y ~ catLetras, data = df)
summary(modeloCatLetras)


Call:
lm(formula = Y ~ catLetras, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-76.300 -25.180   0.539  27.241  74.326 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)   63.372      6.499   9.751 4.63e-16 ***
catLetrasB    -4.053      9.426  -0.430    0.668    
catLetrasC    -1.654      8.769  -0.189    0.851    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 36.76 on 97 degrees of freedom
Multiple R-squared:  0.001916,	Adjusted R-squared:  -0.01866 
F-statistic: 0.09312 on 2 and 97 DF,  p-value: 0.9112


Notemos que catLetras[T.B] es la variable binaria que corresponde a la categoria B de la variable catLetras, y catLetras[T.C] a la categoria C. Ambas tienen su respectivo coeficiente.

Surge la pregunta, ¿Que coeficiente esta asociado al nivel de referencia (la categoria A)? ¿Perdimos la informacion que esa categoria nos daba?

La respuesta es que el "efecto" de la categoria A fue agregado al valor del intercepto $\beta_{0}$, se vera a detalle en el tema _Interpretacion de coeficientes_.

### La variable categorica esta en formato de numero

Supongamos que nos interesa ajustar

$y = \beta_{0}  + \beta_{2}$**catNum**  


Notemos que **catNum** esta representada por numeros (0,1) aunque sea categorica, y ademas cometimos la mala practica de no indicarlo en el preprocesamiento.

Hay dos alternativas, la primera es especificar que es categorica en el preprocesamiento, la segunda es ajustar el modelo **especificando en la formula** que **catNum** es categorica. Esto usando `factor()`

In [7]:
modeloCatNum <- lm(Y ~ factor(catNum), data = df)
summary(modeloCatNum)


Call:
lm(formula = Y ~ factor(catNum), data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-78.724 -25.619   0.424  28.041  72.481 

Coefficients:
                Estimate Std. Error t value Pr(>|t|)    
(Intercept)      61.1639     5.6491   10.83   <2e-16 ***
factor(catNum)1   0.6684     7.4176    0.09    0.928    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 36.61 on 98 degrees of freedom
Multiple R-squared:  8.285e-05,	Adjusted R-squared:  -0.01012 
F-statistic: 0.00812 on 1 and 98 DF,  p-value: 0.9284


C(catNum)[T.1] es la variable binaria correspondiente a la categoria 1 de la variable catNum y tiene su respectivo coeficiente. El efecto de la categoria 0 se ve reflejado en el valor del intercepto $\beta_{0}$

## Transformaciones de las variables


### Funciones comunes

Supongamos que es de interes ajustar $ y = \beta_{0} + \beta_{1}log(x_{1})$

La "formula" del modelo de regresion que se ajusta:

**Y ~ np.log(X1) + X2**

Y se implementa:


In [ ]:
modeloLog = smf.ols(formula='Y ~ np.log(X1) + X2', data=df).fit()


### Funciones personalizadas


Supongamos que es de interes ajustar $ y = \beta_{0} + \beta_{1}log(x_{1} + 100) $

Le estamos realizando una transformacion muy especifica a una variable, por lo que se debe crear la funcion:



In [ ]:
def log_mas_100(x):
  return np.log(x) + 100.0

La "formula" del modelo de regresion que se ajusta:

**Y ~ log_mas_100(X1) + X2**

Y se implementa:

In [ ]:
modeloLog100 = smf.ols(formula='Y ~ log_mas_100(X1) + X2', data=df).fit()


## Interaciones entre variables

Pendiente